# Experiment 1

As the dataset is a tabular data, I'm going to try bunch of experimetns using MLP

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

## Import data

Make train and test features and labels dataframes

In [ ]:
DATA_DIR = "../data"

# --------------  Train Features and Labels ------------------
main_dataset = pd.read_csv(f"{DATA_DIR}/train_features.csv")
main_dataset.set_index("sig_id", inplace=True)  # set sig_id as index

# filter out control samples
main_dataset = main_dataset[main_dataset["cp_type"] == "trt_cp"]
main_dataset["cp_dose_bin"] = main_dataset["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable

# make metadata dataframe
metadata_cols = ["cp_type", "cp_time", "cp_dose"]
main_metadata = main_dataset[metadata_cols].copy()
# train_metadata_df.set_index("sig_id", inplace=True)  # set sig_id as index

# make train_features dataframe
feature_cols = [col for col in main_dataset.columns if col not in metadata_cols and col != "sig_id"]
main_features_df = main_dataset[feature_cols].copy()


# Make train_targets dataframe
main_targets_df = pd.read_csv(f"{DATA_DIR}/train_targets_scored.csv")
main_targets_df.set_index("sig_id", inplace=True)  # set sig_id as index

/tmp/ipykernel_3645798/3466063986.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df["cp_dose_bin"] = train_df["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable


,5-alpha_reductase_inhibitor,11-beta-hsd1_inhibitor,acat_inhibitor,acetylcholine_receptor_agonist,acetylcholine_receptor_antagonist,acetylcholinesterase_inhibitor,adenosine_receptor_agonist,adenosine_receptor_antagonist,adenylyl_cyclase_activator,adrenergic_receptor_agonist,...,tropomyosin_receptor_kinase_inhibitor,trpv_agonist,trpv_antagonist,tubulin_inhibitor,tyrosine_kinase_inhibitor,ubiquitin_specific_protease_inhibitor,vegfr_inhibitor,vitamin_b,vitamin_d_receptor_agonist,wnt_inhibitor
sig_id,,,,,,,,,,,,,,,,,,,,,
id_000644bb2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_000779bfc,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_000a6266a,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_0015fd391,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_001626bd3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
id_fffb1ceed,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_fffb70c0c,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
id_fffc1c3f4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [19]:
DATA_DIR = "../data"

# --------------  Train Features and Labels ------------------
main_dataset = pd.read_csv(f"{DATA_DIR}/train_features.csv")
main_dataset.set_index("sig_id", inplace=True)  # set sig_id as index

# filter out control samples
main_dataset = main_dataset[main_dataset["cp_type"] == "trt_cp"]
main_dataset["cp_dose_bin"] = main_dataset["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable

# make metadata dataframe
metadata_cols = ["cp_type", "cp_time", "cp_dose"]
main_metadata = main_dataset[metadata_cols].copy()
# train_metadata_df.set_index("sig_id", inplace=True)  # set sig_id as index

# make train_features dataframe
feature_cols = [col for col in main_dataset.columns if col not in metadata_cols and col != "sig_id"]
main_features_df = main_dataset[feature_cols].copy()


# Make train_targets dataframe
main_targets_df = pd.read_csv(f"{DATA_DIR}/train_targets_scored.csv")
main_targets_df.set_index("sig_id", inplace=True)  # set sig_id as index

# -------------- Test Features and Labels -----------------
# Make test_features dataframe
test_features_df = pd.read_csv(f"{DATA_DIR}/test_features.csv")
test_features_df.set_index("sig_id", inplace=True)  # set sig_id as index

# filter out control samples
test_features_df = test_features_df[test_features_df["cp_type"] == "trt_cp"]
test_features_df["cp_dose_bin"] = test_features_df["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable

# Make test_metadata dataframe
test_metadata_df = test_features_df[metadata_cols].copy()

# make test_features dataframe
test_features_df = test_features_df[feature_cols].copy()

/tmp/ipykernel_3645798/2558119033.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  main_dataset["cp_dose_bin"] = main_dataset["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable
/tmp/ipykernel_3645798/2558119033.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_features_df["cp_dose_bin"] = test_features_df["cp_dose"].map({"D1": 0, "D2": 1})  # make cp_dose a binary variable


## Train / Val / Dev Split

Split labeled training data into train (~60%), val (~20%), dev (~20%) using `MultilabelStratifiedKFold` (5 folds). Since each fold's held-out indices are non-overlapping and cover the full dataset, we assign fold 0 → val, fold 1 → dev, folds 2–4 → train.

In [20]:
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# Align features and targets on shared sig_ids (main_features_df is already filtered to trt_cp)
common_idx = main_features_df.index.intersection(main_targets_df.index)
X = main_features_df.loc[common_idx]
y = main_targets_df.loc[common_idx]

mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)
splits = list(mskf.split(X, y))

# Each fold's test indices are non-overlapping and together cover the full dataset.
# fold 0 → val (~20%), fold 1 → dev (~20%), folds 2-4 → train (~60%)
val_idx   = splits[0][1]
dev_idx   = splits[1][1]
train_idx = np.concatenate([splits[i][1] for i in range(2, 5)])

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val,   y_val   = X.iloc[val_idx],   y.iloc[val_idx]
X_dev,   y_dev   = X.iloc[dev_idx],   y.iloc[dev_idx]

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Dev: {len(X_dev)}")

Train: 13168 | Val: 4390 | Dev: 4390


## PyTorch Dataset and DataLoaders

In [21]:
class MoADataset(Dataset):
    def __init__(self, features, targets=None):
        self.X = torch.tensor(features.values, dtype=torch.float32)
        self.y = torch.tensor(targets.values, dtype=torch.float32) if targets is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

In [22]:
BATCH_SIZE = 128

train_dataset = MoADataset(X_train, y_train)
val_dataset   = MoADataset(X_val,   y_val)
dev_dataset   = MoADataset(X_dev,   y_dev)
test_dataset  = MoADataset(test_features_df)  # no labels — Kaggle test set

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
dev_loader   = DataLoader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Batches — train: {len(train_loader)} | val: {len(val_loader)} | dev: {len(dev_loader)} | test: {len(test_loader)}")

Batches — train: 103 | val: 35 | dev: 35 | test: 29


## Pytorch Model

In [25]:
class MoAModel(nn.Module):
    def __init__(self, input_dim=X_train.shape[1], output_dim=y_train.shape[1]):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, output_dim)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = torch.relu(self.fc2(x))
        x = self.dropout(x)
        x = torch.sigmoid(self.fc3(x))  # sigmoid for multi-label classification
        return x

## Train Model

Define the hyper-parameters

In [32]:
torch.manual_seed(42)
model = MoAModel()
criterion = nn.BCELoss()  # Binary Cross-Entropy for multi-label classification
optimizer = optim.Adam(model.parameters(), lr=0.001)
n_epochs = 20

Training Loop

In [35]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training started on ", device)

# for epoch in range(n_epochs):
#     for train_X, train_y in train_loader:
        

Training started on  cpu
